# Study tracks on Colab / Kaggle GPU

Runs `edge_ai_compression.experiments.run_track`: trains the baseline seeds a track needs
(skipped if the checkpoint already exists), then runs the track's sweep. Every result is written
to the experiment DB under `results/` (the only results output). Download it at the end and
merge it into your local DB. Numbers never leave the DB by hand.

| Track | Phase | Sweep |
|---|---|---|
| `resnet18_ptq` | 3 | PTQ/QAT ladder, ResNet-18 / CIFAR-10, 3 seeds x 12 rungs |
| `vit_s_ptq` | 3 | SmoothQuant track, ViT-S / CIFAR-10, 3 seeds x 7 rungs |
| `resnet18_prune` | 4 | pruning + recovery ladder, ResNet-18 / CIFAR-10, 3 seeds x 13 rungs |
| `pretrain_variants` | 5 | ResNet-18 x {standard, Quant-Noise, QAT, kurtosis, RigL} x 3 seeds (signals logged) |
| `pretrain_scaling` | 5 | 5 ResNet sizes x {standard, Quant-Noise} x 3 seeds |
| `pretrain_length` | 5 | ResNet-18 w0.5 x {10, 30, 90} epochs x 3 seeds |
| `pretrain_vit` | 5 | ViT t / s / m x 3 seeds |

* Colab: Runtime -> Change runtime type -> GPU. Kaggle: Accelerator -> GPU, Internet on.
* Sweeps **resume**: if the session dies, re-run. Finished variants are skipped (keep
  `results/` and `models/` on Drive, see below).
* Latency columns measured here come from a shared cloud VM (secondary platform, noisy); the
  fingerprint records the CPU. Primary latency numbers come from the M5 Pro.

In [ ]:
BRANCH = "phase5-pretraining"  # latest phase branch
REPO = "https://github.com/eshaan2418/Edge-AI-Model-Compression-Deployment.git"
!git clone --branch $BRANCH $REPO repo
%cd repo
!pip install -q -e ".[kernels,export]"
!bash scripts/build_kernels.sh

In [ ]:
from edge_ai_compression.inference import kernels

print("ISAs:", kernels.supported_isas())

Optional (Colab): persist `results/` and `models/` on Google Drive so an interrupted run resumes.

In [ ]:
import os

if os.path.isdir("/content"):
    from google.colab import drive

    drive.mount("/content/drive")
    root = "/content/drive/MyDrive/edge_ai_tracks"
    for d in ("results", "models"):
        os.makedirs(f"{root}/{d}", exist_ok=True)
        if not os.path.islink(d):
            !rm -rf {d} && ln -s {root}/{d} {d}

Smoke test: same code path, synthetic data, a few seconds.

In [ ]:
SMOKE = "--smoke --seeds 0 --results /tmp/smoke_results --models /tmp/smoke_models"
!python -m edge_ai_compression.experiments.run_track --track resnet18_ptq $SMOKE

Phase 3, ResNet-18 PTQ/QAT ladder (trains 3 seeds x 60 epochs first).

In [ ]:
!python -m edge_ai_compression.experiments.run_track --track resnet18_ptq --device cuda

Phase 3, ViT-S SmoothQuant track (3 seeds x 200 epochs). Check outlier stats first (docs/quantization.md).

In [ ]:
!python -m edge_ai_compression.experiments.run_track --track vit_s_ptq --device cuda

Phase 4, pruning + recovery ladder (reuses the ResNet-18 checkpoints).

In [ ]:
!python -m edge_ai_compression.experiments.run_track --track resnet18_prune --device cuda

Phase 5, compression-aware pre-training variants and signal logging (15 runs).

In [ ]:
!python -m edge_ai_compression.experiments.run_track --track pretrain_variants --device cuda

Phase 5, scaling in model size (30 runs), training length (9 runs), and ViT sizes (9 runs).

In [ ]:
!python -m edge_ai_compression.experiments.run_track --track pretrain_scaling --device cuda
!python -m edge_ai_compression.experiments.run_track --track pretrain_length --device cuda
!python -m edge_ai_compression.experiments.run_track --track pretrain_vit --device cuda

Package the experiment DB for download.

In [ ]:
!zip -qr track_results.zip results
print("Download track_results.zip, unzip it locally, then run:")
print("  python -m edge_ai_compression.experiment_db.merge --src <unzipped>/results --dst results")